In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q01/rewritten/o4_mini_high/checkpoints/post_cell_0_try_0.pickle

trying: ['load_lineitem']
me:  1
trying: ['DATA_ROOT']
me:  0
trying: ['STORAGE_OPTS']
me:  0
trying: ['tpch_parent']
me:  0
trying: ['pd']
me:  0
trying: ['lineitem']
me:  1
Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable tpch_parent
Declaring variable pd
Declaring variable load_lineitem
Declaring variable lineitem


me:  1


Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable tpch_parent
Declaring variable pd
Declaring variable load_lineitem
Declaring variable lineitem


In [4]:
%%RecordEvent
%%time
### cell 1 ###

# Preserve the same Timestamp type for filtering and ensure the grouped result is sorted exactly as in the original
date = pd.Timestamp("1998-09-02")
lineitem_filtered = (
    lineitem.loc[
        lineitem.L_SHIPDATE <= date,
        [
            "L_QUANTITY", "L_EXTENDEDPRICE", "L_DISCOUNT", "L_TAX",
            "L_RETURNFLAG", "L_LINESTATUS", "L_ORDERKEY"
        ]
    ]
    .assign(
        AVG_QTY=lambda df: df.L_QUANTITY,
        AVG_PRICE=lambda df: df.L_EXTENDEDPRICE,
        DISC_PRICE=lambda df: df.L_EXTENDEDPRICE * (1 - df.L_DISCOUNT),
        CHARGE=lambda df: df.L_EXTENDEDPRICE * (1 - df.L_DISCOUNT) * (1 + df.L_TAX)
    )
)

total = (
    lineitem_filtered
    .groupby(["L_RETURNFLAG", "L_LINESTATUS"], as_index=False)
    .agg({
        "L_QUANTITY":   "sum",
        "L_EXTENDEDPRICE": "sum",
        "DISC_PRICE":   "sum",
        "CHARGE":       "sum",
        "AVG_QTY":      "mean",
        "AVG_PRICE":    "mean",
        "L_DISCOUNT":   "mean",
        "L_ORDERKEY":   "count"
    })
    .sort_values(["L_RETURNFLAG", "L_LINESTATUS"])
    .reset_index(drop=True)
)


CPU times: user 104 ms, sys: 56 ms, total: 160 ms
Wall time: 166 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q01/rewritten/o4_mini_high/checkpoints/post_cell_1_try_1.pickle

migration speed (bps): 803708692.538469
---------------------------
variables to migrate:
total 48
lineitem 48
lineitem_filtered 48
tpch_parent 54
load_lineitem 144
DATA_ROOT 80
STORAGE_OPTS 64
pd 72
date 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q01/rewritten/o4_mini_high/checkpoints/post_cell_1_try_1.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['lineitem_filtered', 'date', 'total']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q01/opt_cell_exec_info_1_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[1], f)


In [8]:
opt_output = Out.get(4)

In [ ]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q01/annotated/checkpoints/post_cell_1.pickle
import pdb; pdb.set_trace()
assert compare_df(total_opt, total)

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['DATA_ROOT']
me:  0
trying: ['total']
me:  3
trying: ['orig_output']
me:  2
trying: ['load_lineitem']
me:  1
trying: ['pd']
me:  0
trying: ['gb']
me:  4
trying: ['tpch_parent']
me:  0
trying: ['STORAGE_OPTS']
me:  0
trying: ['lineitem']
me:  1
trying: ['date']
me:  3
trying: ['lineitem_filtered']
me:  3
trying: ['sel']
me:  3
Declaring variable DATA_ROOT
Declaring variable pd
Declaring variable tpch_parent
Declaring variable STORAGE_OPTS
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable orig_output
Declaring variable total
Declaring variable date
Declaring variable lineitem_filtered
Declaring variable sel
Declaring variable gb
--Return--
None
> /var/tmp/ipykernel_127785/3978950369.py(3)<module>()
      1 total_opt = total
      2 get_ipython().run_line_magic('LoadCheckpoint', '/home/dias-benchmarks/tpch/notebooks/q01/annotated/checkpoints/post_cell_1.pickle')
----> 3 import pdb; pdb.set_trace()
      4 assert compare_df(total_opt, total)
      5 


ipdb>  total, total_opt


(  L_RETURNFLAG L_LINESTATUS  L_QUANTITY  L_EXTENDEDPRICE    DISC_PRICE  \
0            A            F    37734107     5.658655e+10  5.375826e+10   
1            N            F      991417     1.487505e+09  1.413082e+09   
2            N            O    74476040     1.117017e+11  1.061182e+11   
3            R            F    37719753     5.656804e+10  5.374129e+10   

         CHARGE    AVG_QTY     AVG_PRICE  L_DISCOUNT  L_ORDERKEY  
0  5.590907e+10  25.522006  38273.129735    0.049985     1478493  
1  1.469649e+09  25.516472  38284.467761    0.050093       38854  
2  1.103670e+11  25.502227  38249.117989    0.049997     2920374  
3  5.588962e+10  25.505794  38250.854626    0.050009     1478870  ,   L_RETURNFLAG L_LINESTATUS  L_QUANTITY  L_EXTENDEDPRICE    DISC_PRICE  \
0            A            F    37734107     5.658655e+10  5.375826e+10   
1            N            F      991417     1.487505e+09  1.413082e+09   
2            N            O    74476040     1.117017e+11  1.061182e+11

ipdb>  total.equals(total_opt)


False


...

   L_RETURNFLAG  L_LINESTATUS  L_QUANTITY  L_EXTENDEDPRICE  DISC_PRICE  \
0          True          True        True            False       False   
1          True          True        True            False       False   
2          True          True        True            False       False   
3          True          True        True            False        True   

   CHARGE  AVG_QTY  AVG_PRICE  L_DISCOUNT  L_ORDERKEY  
0   False     True      False       False        True  
1   False     True      False       False        True  
2   False     True      False       False        True  
3   False     True      False       False        True  


me:  1
trying: ['date']
me:  3
trying: ['lineitem_filtered']


me:  3
trying: ['sel']
me:  3


Declaring variable DATA_ROOT
Declaring variable pd
Declaring variable tpch_parent
Declaring variable STORAGE_OPTS
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable orig_output
Declaring variable total
Declaring variable date
Declaring variable lineitem_filtered
Declaring variable sel
Declaring variable gb


ValueError: Content of df1 and df2 do not match

In [ ]:
total